<!-- HTML file automatically generated from DocOnce source (https://github.com/doconce/doconce/)
doconce format html notebook.do.txt  -->
<!-- dom:TITLE: Lipkin model and PMMs -->

# Lipkin model and PMMs
**Morten Hjorth-Jensen**, Department of Physics, University of Oslo, Norway and the PMM gang

Date: **November 2025**

## Plan for these notes
1. Add notes about the Lipkin model

2. Add notes about the Lipkin model and its coupling to the environment (dissipation and dephasing)

3. Include notes about the time evolution

4. Add notes about entanglement measures

## Code example where we use the Lipkin model and PMMs

In [1]:
%matplotlib inline


import numpy as np
import jax
import jax.numpy as jnp
from jax import grad, jit, lax,vmap
from jax import config
config.update("jax_enable_x64", True)
config.update("jax_platform_name","cpu")

"""#Eigenvalue PMM

implements PMM using jax, and only GD not SGD/Batch
"""

class model(object):
    def __init__(self,
                 n,
                 X_train,
                 y_train,
                 k,
                 l,
                 type=True,
                 X_val=None,
                 y_val=None,
                 param=None,
                 epochs = np.inf,
                 learning_rate =0.01,
                 print_every = 100,
                 verbose: bool = True):

        """
        Initializes the Parameterized Matrix Model (PMM).

        Parameters:
            n (int): The size of the PMM matrix (n x n).
            X_train (jnp.array): Training input features (X data).
            y_train (jnp.array): Training target values (y data, typically k eigenvalues).
            k (int): The number of eigenvalues (initial states) to be predicted.
            l (int): The number of observable matrices (M_1, ..., M_l) in the PMM, where M = M_0 + sum(x_i * M_i).
            type (bool, optional): If True, uses complex parameters (jnp.complex128). If False, uses real parameters (jnp.float64). Defaults to True.
            X_val (jnp.array, optional): Validation input features. Defaults to None.
            y_val (jnp.array, optional): Validation target values. Defaults to None.
            param (jnp.array, optional): Saved initial parameters (theta) to use instead of generating new ones. Defaults to None.
            epochs (int, optional): The maximum number of iterations for gradient descent (GD). Defaults to infinity (np.inf).
            learning_rate (float, optional): The learning rate for the ADAM optimizer. Defaults to 0.01.
            print_every (int, optional): Number of GD iterations before printing the loss. Defaults to 100.
            verbose (bool, optional): Enable or disable printing the loss function during training. Defaults to True.
        """

        self.n = n #size of PMM
        self.y_train = y_train # y train set
        self.X_train = X_train #x train set
        self.k = k #number of eigenvalues
        self.l= l #number of M matrices
        self.y_val = y_val #y validation set
        self.X_val = X_val #x validatoin set
        self.params = param #to used saved parameters
        self.epochs = epochs #number of iteration for gd
        self.learning_rate = learning_rate #learning rate
        self.print_every = print_every #how many iterations of gd before pringing loss
        self.verbose = verbose #enable for printing loss function
        self.type = type #complex or real
        if self.type == True:
            self.dtype = jnp.complex128
        elif self.type == False:
            self.dtype = jnp.float64

    ####################################
    # 1. GETTING THE PARAMETERS
    ####################################
    def get_param(self):
      """
      Generates initial random parameters (theta) for the PMM.

      The total number of parameters is (l + 1) * n * n, corresponding to
      the l+1 matrices: M_0 and the l observable matrices M_1 to M_l.

      Returns:
          jnp.array: A flat array of randomly initialized parameters (theta).
                      Complex if self.type is True, real otherwise.
      """
      num_params =  (self.l+1)*self.n*self.n
      # Generate random parameters
      if self.type == True:
          theta = 0.01*(np.random.randn(num_params) + 1j*np.random.randn(num_params))
      elif self.type == False:
          theta = 0.01*(np.random.randn(num_params))
      return theta

    ####################################
    # 2. SPLITTING THE PARAMETERS
    ####################################
    def split_params(self, theta):
      """
      Reshapes the flat parameter array (theta) into the PMM matrices O_0, O_1, ..., O_l.
      It also enforces Hermitian symmetry for the matrices O_i by averaging with their conjugate transpose.

      Parameters:
          theta (jnp.array): A flat array of parameters.

      Returns:
          jnp.array: A 3D array of dimension (l+1, n, n) containing the Hermitian PMM matrices.
                     The first slice O[0] corresponds to M_0, and O[1:] corresponds to M_1 to M_l.
      """
      theta = jnp.asarray(theta)
      # O is split into O_0 (1st matrix) and O_1...O_l (l matrices)
      # O = [M_0, M_1, ..., M_l] (l+1 matrices in total)
      O = jnp.reshape(theta,(self.l+1,self.n,self.n))
      O = (O + O.conj().transpose(0,2,1))/2
      return O


    ####################################
    # 3. PMM Model
    ####################################
    def update(self, theta, X_data):
      """
      Calculates the predicted eigenvalues (y_pred) for a given set of parameters (theta)
      and input features (X_data).

      The PMM matrix M is constructed as: $M = M_0 + \sum_{i=1}^l x_i M_i$.
      The output is the first k eigenvalues of the constructed matrix M.

      Parameters:
          theta (jnp.array): A flat array of model parameters.
          X_data (jnp.array): Input features, where each row $x$ corresponds to a data point.

      Returns:
          jnp.array: The predicted values (first k eigenvalues) for each data point in X_data.
      """
      O = self.split_params(theta)
      def pmm_ev(x,theta,O):
        #M = M_0 + \sum_i^N x_i M_i
        # The einsum operation performs M = O[0] + sum(x_i * O_i)
        # Note: The indexing in einsum suggests O[1:] are M_i and O[0] is M_0,
        M = O[0] + jnp.einsum('i,ijk->jk',x,O[1:])
        E = jnp.linalg.eigvalsh(M)
        return E[:self.k]
      # Vectorize the PMM evaluation across all data points in X_data
      y_pred = vmap(pmm_ev,in_axes=(0,None,None))(X_data,theta,O)
      return y_pred


    def Cost(self,y_true,y_pred):
      """
      Calculates the mean squared error (MSE) between true and predicted values.

      Cost = mean(|y_true - y_pred|^2)

      Parameters:
          y_true (jnp.array): The true target values.
          y_pred (jnp.array): The predicted values.

      Returns:
          float: The real-valued mean squared loss.
      """
      cost = jnp.mean((abs(y_true - y_pred)**2))
      return cost.real


    def Loss(self,theta,X_data,y_true):
      """
      Calculates the full loss function: first computes predictions, then the cost.

      Parameters:
          theta (jnp.array): Model parameters.
          X_data (jnp.array): Input features.
          y_true (jnp.array): True target values.

      Returns:
          float: The real-valued loss (Cost) for the given parameters and data.
      """
      y_pred = self.update(theta,X_data)
      return self.Cost(y_true,y_pred)


    def dLossdx(self,theta,X_data,y_true):
      """
      Computes the gradient of the Loss function with respect to the parameters (theta)
      using JAX's automatic differentiation.

      Parameters:
          theta (jnp.array): Model parameters.
          X_data (jnp.array): Input features.
          y_true (jnp.array): True target values.

      Returns:
          jnp.array: The gradient of the loss function with respect to theta.
      """
      return grad(self.Loss)(theta,X_data,y_true)

    def ADAM(self,g,i,m,v):
      """
      Performs one step of the ADAM (Adaptive Moment Estimation) optimization algorithm.

      Parameters:
          g (jnp.array): The current gradient of the loss function.
          i (int): The current iteration number (0-indexed).
          m (jnp.array): The previous biased first moment vector (m_{t-1}).
          v (jnp.array): The previous biased second raw moment vector (v_{t-1}).

      Returns:
          tuple: (delta_theta, m_new, v_new) where:
                  - delta_theta (jnp.array): The parameter update direction.
                  - m_new (jnp.array): The updated first moment vector (m_t).
                  - v_new (jnp.array): The updated second moment vector (v_t).
      """
      b1=0.9
      b2=0.999
      eps=1e-8
      # update biased first moment estimate
      m_new = b1 * m + (1 - b1) * g
      # update biased second raw moment estimate
      v_new = b2 * v + (1 - b2) * (g*g.conj())
      # compute bias-corrected first moment estimate
      mhat = m_new / (1 - b1 ** (i + 1))
      # compute bias-corrected second raw moment estimate
      vhat = v_new / (1 - b2 ** (i + 1))
      delta_theta = mhat / (jnp.sqrt(vhat) + eps)
      return delta_theta, m_new, v_new


    def train(self):
      """
      Executes the training process using gradient descent with the ADAM optimizer.

      The model iterates through a specified number of epochs, computes the gradient
      of the loss, updates parameters using ADAM, and monitors the loss. It saves
      the parameters that result in the best validation loss (if X_val is provided)
      or the best training loss otherwise.

      Returns:
          jnp.array: The best set of parameters (theta) found during training.
      """
      theta = jnp.array(self.get_param() if self.params is None else self.params,dtype=self.dtype)

      self.best_param = None
      best_loss = np.inf
      best_val_loss = np.inf

      X_train = self.X_train
      y_train = self.y_train
      X_val = self.X_val
      y_val = self.y_val

      # JAX Optimization: jit compiles the core mathematical operations (gradient,
      # loss, PMM matrix construction, and eigenvalue computation) into a single,
      # optimized XLA kernel for maximum speed.
      # (dLossdx -> Loss -> update -> split_params) that uses JAX arrays.
      jgrad = jit(lambda theta,X_train,y_train: self.dLossdx(theta,X_train,y_train))
      jLoss = jit(lambda theta,X_train,y_train: self.Loss(theta,X_train,y_train))
      jADAM = jit(lambda g_in, i_in, m_in, v_in: self.ADAM(g_in, i_in, m_in, v_in))

      #ADAM parameters
      m = np.zeros(theta.shape, dtype=theta.dtype)
      v = np.zeros(theta.shape, dtype=theta.dtype)
      i = 0
      num_samples = X_train.shape[0]
      while i < self.epochs:
          #calculate gradient
          g = jgrad(theta,X_train,y_train)
          delta_theta, m, v = jADAM(g.conj(),i,m,v)
          theta = theta - self.learning_rate*delta_theta

          #print loss
          loss = jLoss(theta,X_train,y_train)

          ################################################
          # for verbose printing
          ################################################
          if self.verbose and i % self.print_every == 0:
              # ANSI escape codes for colors and bold text
              COLOR_CYAN = "\033[96m"
              COLOR_GREEN = "\033[92m"
              COLOR_YELLOW = "\033[93m"
              COLOR_RED = "\033[91m"
              COLOR_RESET = "\033[0m" # Resets all formatting

              # Determine color based on loss trend (conceptual, you'd need history for this)
              # For simplicity, let's just make the losses stand out.

              # Header Line
              print(f"{COLOR_CYAN}═════════════════════════════════════════════════{COLOR_RESET}")

              # Epoch line
              print(f"  {COLOR_CYAN}🌌 Epoch {i:<8}{COLOR_RESET} | {COLOR_GREEN}Training Loss: {loss:.2e}{COLOR_RESET} |", end="")

              # Validation Loss line (if applicable)
              if X_val is not None:
                  val_loss = jLoss(theta, X_val, y_val)
                  print(f" {COLOR_YELLOW}Validation Loss: {val_loss:.2e}{COLOR_RESET} |", end="")

              print(f"\n{COLOR_CYAN}═════════════════════════════════════════════════{COLOR_RESET}")
          ################################################

          i += 1

          if X_val is not None:
              val_loss = jLoss(theta,X_val,y_val)
              if val_loss < best_val_loss:
                  self.best_param = theta
                  best_val_loss = val_loss
          elif loss < best_loss:
              self.best_param = theta
              best_loss = loss

      if X_val is not None:
          return self.best_param
      else:
          return self.best_param

    def predict(self,theta,X_test):
      """
      Generates predictions (eigenvalues) on new test data using a trained set of parameters.

      Parameters:
          theta (jnp.array): The trained parameters (often the best_param returned by `train`).
          X_test (jnp.array): The input features (X data) for prediction.

      Returns:
          jnp.array: The predicted values (first k eigenvalues) for X_test.
      """
      return self.update(theta,X_test)

# how to use
"""
x_train = np.random.rand(100,5)
y_train = np.random.rand(100,5)
x_test = np.random.rand(100,5)
y_test = np.random.rand(100,5)
x_val = None
y_val = None
n = 5
k = 1
l = 1
type = True #True: Complex

Model = model(n,
              x_train,
              y_train,
              k,
              l,
              type,
              x_val,
              y_val,
              param=None,
              epochs = np.inf,
              learning_rate =0.01,
              print_every = 100,
              verbose=True)

New_theta = Model.train()

y_pred = Model.predict(New_theta,x_test)
"""

"""##LMG"""

import math
from itertools import combinations
from matplotlib import pyplot as plt

class ManyBodyModel:
    """
    Abstract base for a parametric Hamiltonian H(g),
    where g is a scalar control parameter (V, G, ...).
    """

    def __init__(self, dim, name="Model"):
        self.dim = dim
        self.name = name

    def hamiltonian(self, g):
        """
        Return the Hamiltonian H(g) as a dim x dim numpy array.
        Must be implemented by subclasses.
        """
        raise NotImplementedError

    def spectrum(self, grid, k):
        """
        Compute lowest k eigenvalues for each value in grid.

        Parameters
        ----------
        grid : array-like
            1D array of control parameters g.
        k : int
            Number of lowest eigenvalues to return.

        Returns
        -------
        E : ndarray, shape (len(grid), k)
        """
        from numpy.linalg import eigvalsh
        E = []
        for g in grid:
            H = self.hamiltonian(g)
            evals = eigvalsh(H)
            E.append(evals[:k])
        return np.array(E)


# ============================================================
# Lipkin model
# ============================================================

class LipkinModel(ManyBodyModel):
    """
    Lipkin-Meshkov-Glick model:

        H = epsilon * Jz + (V/N) * (Jx^2 - Jy^2)

    where (Jx, Jy, Jz) are collective spin operators.

    Can work in:
      - full 2^N Hilbert space
      - or symmetric J = N/2 sector (dim = N+1)
    """

    def __init__(self, N, epsilon=1.0, use_symmetric_sector=True):
        self.N = N
        self.epsilon = epsilon
        self.use_symmetric_sector = use_symmetric_sector

        if use_symmetric_sector:
            Jx, Jy, Jz = self._build_collective_operators_symmetric(N)
            dim = Jx.shape[0]  # N+1
            name = f"Lipkin (symmetric J=N/2, N={N})"
        else:
            Jx, Jy, Jz = self._build_collective_operators_full(N)
            dim = Jx.shape[0]  # 2^N
            name = f"Lipkin (full space, N={N})"

        super().__init__(dim=dim, name=name)

        self.Jx = Jx
        self.Jy = Jy
        self.Jz = Jz

        # Precompute H0 and Hint: H(V) = H0 + V*Hint
        self.H0 = epsilon * self.Jz
        self.Hint = (1.0 / N) * (self.Jx @ self.Jx - self.Jy @ self.Jy)

    # -- public API --

    def hamiltonian(self, V):
        """
        Lipkin Hamiltonian for interaction strength V.
        """
        return self.H0 + V * self.Hint

    # -- internal helpers --

    @staticmethod
    def _build_collective_operators_full(N):
        """
        Build Jx, Jy, Jz in the full 2^N Hilbert space
        from tensor products of single-qubit Pauli matrices.
        """
        dim = 2**N
        sx = np.array([[0, 1],
                       [1, 0]], dtype=np.complex128)
        sy = np.array([[0, -1j],
                       [1j, 0]], dtype=np.complex128)
        sz = np.array([[1, 0],
                       [0, -1]], dtype=np.complex128)
        id2 = np.eye(2, dtype=np.complex128)

        Jx = np.zeros((dim, dim), dtype=np.complex128)
        Jy = np.zeros((dim, dim), dtype=np.complex128)
        Jz = np.zeros((dim, dim), dtype=np.complex128)

        for site in range(N):
            ops = []
            for pos in range(N):
                if pos == site:
                    ops.append((sx, sy, sz))
                else:
                    ops.append((id2, id2, id2))

            sx_i = ops[0][0]
            sy_i = ops[0][1]
            sz_i = ops[0][2]
            for pos in range(1, N):
                sx_i = np.kron(sx_i, ops[pos][0])
                sy_i = np.kron(sy_i, ops[pos][1])
                sz_i = np.kron(sz_i, ops[pos][2])

            Jx += 0.5 * sx_i
            Jy += 0.5 * sy_i
            Jz += 0.5 * sz_i

        return Jx, Jy, Jz

    @staticmethod
    def _build_collective_operators_symmetric(N):
        """
        Build Jx, Jy, Jz in the fully symmetric total-spin sector J = N/2.

        Basis: |J, M>, M = J, J-1, ..., -J. Dimension = N+1.
        """
        J = N / 2.0
        dim = int(2 * J + 1)

        M_vals = np.arange(J, -J - 1, -1, dtype=float)

        Jp = np.zeros((dim, dim), dtype=np.complex128)
        Jm = np.zeros((dim, dim), dtype=np.complex128)
        Jz = np.zeros((dim, dim), dtype=np.complex128)

        for i, M in enumerate(M_vals):
            # J_z |J,M> = M |J,M>
            Jz[i, i] = M

            # J_+ |J,M> -> |J,M+1>, index i-1
            if i > 0:
                coef = math.sqrt(J * (J + 1.0) - M * (M + 1.0))
                Jp[i - 1, i] = coef

            # J_- |J,M> -> |J,M-1>, index i+1
            if i < dim - 1:
                coef = math.sqrt(J * (J + 1.0) - M * (M - 1.0))
                Jm[i + 1, i] = coef

        Jx = 0.5 * (Jp + Jm)
        Jy = -0.5j * (Jp - Jm)

        return Jx, Jy, Jz

# ------------------------
# LIPKIN EXAMPLE
# ------------------------
N = 20
epsilon_lip = 1.0
k_levels = 3

use_symmetric = True  # True: J=N/2 block; False: full 2^N
lipkin = LipkinModel(N=N, epsilon=epsilon_lip,
                      use_symmetric_sector=use_symmetric)
print(f"{lipkin.name}, dim = {lipkin.dim}")

V_train = np.linspace(0.0, 2.0, 20)
V_test = np.linspace(0.0, 2.0, 100)

E_train_lip = lipkin.spectrum(V_train, k=k_levels)
E_test_lip = lipkin.spectrum(V_test, k=k_levels)

# Plot exact Lipkin spectrum
plt.figure(figsize=(8, 5))
for ell in range(k_levels):
    plt.plot(V_test, E_test_lip[:, ell], lw=2,
              label=f"Exact Lipkin E_{ell}")
plt.scatter(V_train, E_train_lip[:, 0], color="k", s=40,
            label="Train points (ground)")
plt.xlabel("V")
plt.ylabel("E_n(V)")
plt.title(f"Lipkin spectrum (N={N}, dim={lipkin.dim})")
plt.legend()
plt.tight_layout()
plt.show()

"""##Train PMM on LMG"""

X_train = V_train.reshape(-1,1)
y_train = E_train_lip
x_val = None
y_val = None
n = 5
k = 3
l = 1
type = True #True: Complex

Model = model(n,
              X_train,
              y_train,
              k,
              l,
              type,
              x_val,
              y_val,
              param=None,
              epochs = 40000,
              learning_rate =0.01,
              print_every = 5000,
              verbose=True)

New_theta = Model.train()

V_test = V_test.reshape(-1,1)
y_pred = Model.predict(New_theta,V_test)
plt.figure(figsize=(8, 5))
for ell in range(k_levels):
    plt.plot(V_test, E_test_lip[:, ell], lw=2,
              label=f"Exact Lipkin E_{ell}")
    plt.plot(V_test, y_pred[:, ell], "--", lw=2,
              label=f"Reduced PMM E_{ell}")
plt.xlabel("V")
plt.ylabel("E_n(V)")
plt.title(f"Lipkin: reduced PMM vs exact (n_eff={n}, dim={lipkin.dim})")
plt.legend()
plt.tight_layout()
plt.show()

#plot abs error
plt.figure(figsize=(8, 5))
for ell in range(k_levels):
    plt.plot(V_test, abs(E_test_lip[:, ell] - y_pred[:, ell]), lw=2,label=f"E_{ell}")
plt.xlabel("V")
plt.ylabel("E_n(V)")
plt.title(f"Lipkin: abs error (n_eff={n}, dim={lipkin.dim})")
plt.yscale("log")
plt.legend()
plt.tight_layout()
plt.show()

"""#Time Evolution PMM

"""

class model2(object):
    def __init__(self,
                 n,
                 X_train,
                 y_train,
                 k,
                 l,
                 #r,
                 type=True,
                 X_val=None,
                 y_val=None,
                 param=None,
                 epochs = np.inf,
                 learning_rate =0.01,
                 print_every = 100,
                 verbose: bool = True):

        """
        Initializes the Parameterized Matrix Model (PMM).

        Parameters:
            n (int): The size of the PMM matrix (n x n).
            X_train (jnp.array): Training input features (X data).
            y_train (jnp.array): Training target values (y data, typically k eigenvalues).
            k (int): The number of eigenvalues (initial states) to be predicted.
            l (int): The number of hamiltonian matrices (M_1, ..., M_l) in the PMM, where M = M_0 + sum(x_i * M_i).
            r (int): The number of observable matrices (O_1, ..., O_r) in the PMM
            type (bool, optional): If True, uses complex parameters (jnp.complex128). If False, uses real parameters (jnp.float64). Defaults to True.
            X_val (jnp.array, optional): Validation input features. Defaults to None.
            y_val (jnp.array, optional): Validation target values. Defaults to None.
            param (jnp.array, optional): Saved initial parameters (theta) to use instead of generating new ones. Defaults to None.
            epochs (int, optional): The maximum number of iterations for gradient descent (GD). Defaults to infinity (np.inf).
            learning_rate (float, optional): The learning rate for the ADAM optimizer. Defaults to 0.01.
            print_every (int, optional): Number of GD iterations before printing the loss. Defaults to 100.
            verbose (bool, optional): Enable or disable printing the loss function during training. Defaults to True.
        """

        self.n = n #size of PMM
        self.y_train = y_train # y train set
        self.X_train = X_train #x train set
        self.k = k #number of initial states
        self.l= l #number of M matrices
        #self.r = r #number of O observables
        self.y_val = y_val #y validation set
        self.X_val = X_val #x validatoin set
        self.params = param #to used saved parameters
        self.epochs = epochs #number of iteration for gd
        self.learning_rate = learning_rate #learning rate
        self.print_every = print_every #how many iterations of gd before pringing loss
        self.verbose = verbose #enable for printing loss function
        self.type = type #complex or real
        if self.type == True:
            self.dtype = jnp.complex128
        elif self.type == False:
            self.dtype = jnp.float64

    ####################################
    # 1. GETTING THE PARAMETERS
    ####################################
    def get_param(self):
        """
        Generates initial random parameters (theta) for the PMM.

        The total number of parameters is (l + 1) * n * n + r * n * n,

        (l + 1) * n * ncorresponding to the l+1 matrices: M_0 and the
        l M_i matrices M_1 to M_l.

        r * n * n corresponding to the r O observables.

        Returns:
            jnp.array: A flat array of randomly initialized parameters (theta).
                       Complex if self.type is True, real otherwise.
        """
        num_params =  (self.l+1)*self.n*self.n #+ self.r*self.n*self.n
        # Generate random parameters
        if self.type == True:
            theta = 0.01*(np.random.randn(num_params) + 1j*np.random.randn(num_params))
        elif self.type == False:
            theta = 0.01*(np.random.randn(num_params))
        return theta

    ####################################
    # 2. SPLITTING THE PARAMETERS
    ####################################
    def split_params(self, theta):
      """
      Reshapes the flat parameter array (theta) into the PMM matrices O_0, O_1, ..., O_l.
      It also enforces Hermitian symmetry for the matrices O_i by averaging with their conjugate transpose.

      Parameters:
          theta (jnp.array): A flat array of parameters.

      Returns:
          jnp.array: A 3D array of dimension (l+1, n, n) containing the Hermitian PMM matrices.
                     The first slice O[0] corresponds to M_0, and O[1:] corresponds to M_1 to M_l.
      """
      theta = jnp.asarray(theta)
      # M is split into M_0 (1st matrix) and M_1...M_l (l matrices)
      # M = [M_0, M_1, ..., M_l] (l+1 matrices in total)
      M = jnp.reshape(theta[:(self.l+1)*self.n*self.n],(self.l+1,self.n,self.n))
      M = (M + M.conj().transpose(0,2,1))/2
      # O is split into O_0...M_r (r matrices)
      #O = jnp.reshape(theta[(self.l+1)*self.n*self.n:],(self.r,self.n,self.n))
      #O = (O + O.conj().transpose(0,2,1))/2
      return M#,O


    ####################################
    # 3. PMM Model
    ####################################
    def update(self, theta, X_data):
      """
      Performs quantum time evolution and calculates the expectation values
      of a set of observables for each input data point $X$.

      The input features X are assumed to contain coupling constants/system parameters
      and time $t$. The system Hamiltonian $H$ is constructed, time-evolved using
      the phase factor $e^{-iEt}$, and then used to compute observable expectation
      values.

      Parameters:
          theta (jnp.array): A flat array of model parameters used to define the
                             Hamiltonian $H$ and the Observable matrices $O$.
          X_data (jnp.array): Input features, where each row $x = [c_1, c_2, ..., t]$
                              contains system parameters (c) and the evolution time (t).

      Returns:
          jnp.array: The predicted expectation values of the observables for each
                     data point in X_data.
      """
      #M,O = self.split_params(theta)
      M = self.split_params(theta)
      def pmm_ev(x,theta,O):
        """
        Calculates the time-evolved state $\psi_t$ and the expectation value
        of the Observable $O$ for a single input $x$.
        """
        # The structure of x is assumed to be: x[0] = system parameter, x[1] = time t.
        # The variable 'c' is x[0] and 't' is x[1]
        c = x[0]
        t = x[1]
        c = c.reshape(1,)

        # Construct the Hamiltonian H from the PMM definition: $H = M_0 + \sum_{i=1}^l c_i M_i$
        H = M[0] + jnp.einsum('i,ijk->jk',c,M[1:])

        # 2. Diagonalization of the Hamiltonian
        # E: Eigenvalues (Energy levels, $E_n$). V: Eigenvectors (Stationary states, $|n\rangle$).
        # jnp.linalg.eigh is used for Hermitian matrices, ensuring real eigenvalues.
        E, V = jnp.linalg.eigh(H)

        # 3. Time Evolution Operator
        # Compute the phase factor: $e^{-i E_n t}$ for all $n$ energy levels.
        phase = jnp.exp(-1j * E * t)

        # Compute the full time evolution operator (U(t) = exp(-iHt)).
        # $U(t) = V \cdot \text{Diag}(e^{-i E_n t}) \cdot V^\dagger$
        expm = V @ jnp.diag(phase) @ V.conj().T

        # 4. Initial State Definition
        # $\psi_0$: Define the initial state as a superposition (or subspace)
        # spanned by the first 'k' eigenvectors of the Hamiltonian.
        # V[:,:self.k] selects the first 'k' columns (eigenvectors) of V.
        psi_0 = V[:,:self.k]

        # 5. Time-Evolved State
        # Compute the time-evolved state: $\psi_t = U(t) \psi_0$
        # Note: Since psi_0 is a matrix (subspace), psi_t is also a matrix.
        psi_t = expm @ psi_0

        # 6. Observable Expectation Value
        # Calculate the expectation value $\langle O \rangle$.
        # Since psi_t is a $n \times k$ matrix, this operation computes $k \times k$
        # expectation values: $\langle \psi_{t,i} | O | \psi_{t,j} \rangle$.
        # O here is assumed to be a single $n \times n$ Observable matrix (M[0]).
        Obv = psi_t.conj().T @ O @ psi_t

        # The result is reshaped to a flat vector, likely to match the expected output format.
        # This implies the model uses the elements of the $k \times k$ matrix as output targets.
        return Obv.reshape(1,)
      O = M[0]
      # Vectorize the time evolution (pmm_ev) across all data points in X_data
      y_pred = vmap(pmm_ev,in_axes=(0,None,None))(X_data,theta,O)
      return y_pred


    def Cost(self,y_true,y_pred):
      """
      Calculates the mean squared error (MSE) between true and predicted values.

      Cost = mean(|y_true - y_pred|^2)

      Parameters:
          y_true (jnp.array): The true target values.
          y_pred (jnp.array): The predicted values.

      Returns:
          float: The real-valued mean squared loss.
      """
      cost = jnp.mean((abs(y_true - y_pred)**2))
      return cost.real


    def Loss(self,theta,X_data,y_true):
      """
      Calculates the full loss function: first computes predictions, then the cost.

      Parameters:
          theta (jnp.array): Model parameters.
          X_data (jnp.array): Input features.
          y_true (jnp.array): True target values.

      Returns:
          float: The real-valued loss (Cost) for the given parameters and data.
      """
      y_pred = self.update(theta,X_data)
      return self.Cost(y_true,y_pred)


    def dLossdx(self,theta,X_data,y_true):
        """
        Computes the gradient of the Loss function with respect to the parameters (theta)
        using JAX's automatic differentiation.

        Parameters:
            theta (jnp.array): Model parameters.
            X_data (jnp.array): Input features.
            y_true (jnp.array): True target values.

        Returns:
            jnp.array: The gradient of the loss function with respect to theta.
        """
        return grad(self.Loss)(theta,X_data,y_true)

    def ADAM(self,g,i,m,v):
        """
        Performs one step of the ADAM (Adaptive Moment Estimation) optimization algorithm.

        Parameters:
            g (jnp.array): The current gradient of the loss function.
            i (int): The current iteration number (0-indexed).
            m (jnp.array): The previous biased first moment vector (m_{t-1}).
            v (jnp.array): The previous biased second raw moment vector (v_{t-1}).

        Returns:
            tuple: (delta_theta, m_new, v_new) where:
                   - delta_theta (jnp.array): The parameter update direction.
                   - m_new (jnp.array): The updated first moment vector (m_t).
                   - v_new (jnp.array): The updated second moment vector (v_t).
        """
        b1=0.9
        b2=0.999
        eps=1e-8
        # update biased first moment estimate
        m_new = b1 * m + (1 - b1) * g
        # update biased second raw moment estimate
        v_new = b2 * v + (1 - b2) * (g*g.conj())
        # compute bias-corrected first moment estimate
        mhat = m_new / (1 - b1 ** (i + 1))
        # compute bias-corrected second raw moment estimate
        vhat = v_new / (1 - b2 ** (i + 1))
        delta_theta = mhat / (jnp.sqrt(vhat) + eps)
        return delta_theta, m_new, v_new


    def train(self):
      """
      Executes the training process using gradient descent with the ADAM optimizer.

      The model iterates through a specified number of epochs, computes the gradient
      of the loss, updates parameters using ADAM, and monitors the loss. It saves
      the parameters that result in the best validation loss (if X_val is provided)
      or the best training loss otherwise.

      Returns:
          jnp.array: The best set of parameters (theta) found during training.
      """
      theta = jnp.array(self.get_param() if self.params is None else self.params,dtype=self.dtype)

      self.best_param = None
      best_loss = np.inf
      best_val_loss = np.inf

      X_train = self.X_train
      y_train = self.y_train
      X_val = self.X_val
      y_val = self.y_val

      # JAX Optimization: jit compiles the core mathematical operations (gradient,
      # loss, PMM matrix construction, and eigenvalue computation) into a single,
      # optimized XLA kernel for maximum speed.
      # (dLossdx -> Loss -> update -> split_params) that uses JAX arrays.
      jgrad = jit(lambda theta,X_train,y_train: self.dLossdx(theta,X_train,y_train,))
      jLoss = jit(lambda theta,X_train,y_train: self.Loss(theta,X_train,y_train,))
      jADAM = jit(lambda g_in, i_in, m_in, v_in: self.ADAM(g_in, i_in, m_in, v_in))

      #ADAM parameters
      m = np.zeros(theta.shape, dtype=theta.dtype)
      v = np.zeros(theta.shape, dtype=theta.dtype)
      i = 0
      num_samples = X_train.shape[0]
      while i < self.epochs:
          #calculate gradient
          g = jgrad(theta,X_train,y_train)
          delta_theta, m, v = jADAM(g.conj(),i,m,v)
          theta = theta - self.learning_rate*delta_theta

          #print loss
          loss = jLoss(theta,X_train,y_train)

          ################################################
          # for verbose printing
          ################################################
          if self.verbose and i % self.print_every == 0:
              # ANSI escape codes for colors and bold text
              COLOR_CYAN = "\033[96m"
              COLOR_GREEN = "\033[92m"
              COLOR_YELLOW = "\033[93m"
              COLOR_RED = "\033[91m"
              COLOR_RESET = "\033[0m" # Resets all formatting

              # Determine color based on loss trend (conceptual, you'd need history for this)
              # For simplicity, let's just make the losses stand out.

              # Header Line
              print(f"{COLOR_CYAN}═════════════════════════════════════════════════{COLOR_RESET}")

              # Epoch line
              print(f"  {COLOR_CYAN}🌌 Epoch {i:<8}{COLOR_RESET} | {COLOR_GREEN}Training Loss: {loss:.2e}{COLOR_RESET} |", end="")

              # Validation Loss line (if applicable)
              if X_val is not None:
                  val_loss = jLoss(theta, X_val, y_val)
                  print(f" {COLOR_YELLOW}Validation Loss: {val_loss:.2e}{COLOR_RESET} |", end="")

              print(f"\n{COLOR_CYAN}═════════════════════════════════════════════════{COLOR_RESET}")
          ################################################

          i += 1

          if X_val is not None:
              val_loss = jLoss(theta,X_val,y_val)
              if val_loss < best_val_loss:
                  self.best_param = theta
                  best_val_loss = val_loss
          elif loss < best_loss:
              self.best_param = theta
              best_loss = loss

      if X_val is not None:
          return self.best_param
      else:
          return self.best_param

    def predict(self,theta,X_test):
      """
      Generates predictions (eigenvalues) on new test data using a trained set of parameters.

      Parameters:
          theta (jnp.array): The trained parameters (often the best_param returned by `train`).
          X_test (jnp.array): The input features (X data) for prediction.

      Returns:
          jnp.array: The predicted values (first k eigenvalues) for X_test.
      """
      return self.update(theta,X_test)

"""##LMG TIME"""

import math
import numpy as np
from itertools import combinations
from matplotlib import pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim


# ============================================================
# Base class for many-body models (NumPy)
# ============================================================

class ManyBodyModel:
    """
    Abstract base for a parametric Hamiltonian H(g),
    where g is a scalar control parameter (V, G, ...).
    """

    def __init__(self, dim, name="Model"):
        self.dim = dim
        self.name = name

    # ------------- core API -------------

    def hamiltonian(self, g):
        """
        Return the Hamiltonian H(g) as a dim x dim numpy array.
        Must be implemented by subclasses.
        """
        raise NotImplementedError

    def spectrum(self, grid, k):
        """
        Compute lowest k eigenvalues for each value in grid.

        Returns: E (len(grid), k)
        """
        from numpy.linalg import eigvalsh
        E = []
        for g in grid:
            H = self.hamiltonian(g)
            evals = eigvalsh(H)
            E.append(evals[:k])
        return np.array(E)

    # ------------- time evolution (NumPy) -------------

    def time_evolution(self, g, psi0, times):
        """
        Exact time evolution |psi(t)> = exp(-i H(g) t) |psi0>.
        """
        from numpy.linalg import eigh
        H = self.hamiltonian(g)
        evals, evecs = eigh(H)
        coeffs = evecs.conj().T @ psi0

        states = []
        for t in times:
            phase = np.exp(-1j * evals * t)
            psi_t = evecs @ (phase * coeffs)
            states.append(psi_t)
        return np.array(states)

    def observable_time_series(self, g, psi0, O, times):
        """
        <O>(t) for each t in times.
        """
        psi_t = self.time_evolution(g, psi0, times)
        vals = []
        for psi in psi_t:
            vals.append(np.vdot(psi, O @ psi))
        return np.array(vals)


# ============================================================
# Lipkin model (NumPy)
# ============================================================

class LipkinModel(ManyBodyModel):
    """
    Lipkin-Meshkov-Glick model:

        H = epsilon * Jz + (V/N) * (Jx^2 - Jy^2)
    """

    def __init__(self, N, epsilon=1.0, use_symmetric_sector=True):
        self.N = N
        self.epsilon = epsilon
        self.use_symmetric_sector = use_symmetric_sector

        if use_symmetric_sector:
            Jx, Jy, Jz = self._build_collective_operators_symmetric(N)
            dim = Jx.shape[0]  # N+1
            name = f"Lipkin (symmetric J=N/2, N={N})"
        else:
            Jx, Jy, Jz = self._build_collective_operators_full(N)
            dim = Jx.shape[0]  # 2^N
            name = f"Lipkin (full space, N={N})"

        super().__init__(dim=dim, name=name)

        self.Jx = Jx
        self.Jy = Jy
        self.Jz = Jz

        self.H0 = epsilon * self.Jz
        self.Hint = (1.0 / N) * (self.Jx @ self.Jx - self.Jy @ self.Jy)

    def hamiltonian(self, V):
        return self.H0 + V * self.Hint

    @staticmethod
    def _build_collective_operators_full(N):
        dim = 2**N
        sx = np.array([[0, 1],
                       [1, 0]], dtype=np.complex128)
        sy = np.array([[0, -1j],
                       [1j, 0]], dtype=np.complex128)
        sz = np.array([[1, 0],
                       [0, -1]], dtype=np.complex128)
        id2 = np.eye(2, dtype=np.complex128)

        Jx = np.zeros((dim, dim), dtype=np.complex128)
        Jy = np.zeros((dim, dim), dtype=np.complex128)
        Jz = np.zeros((dim, dim), dtype=np.complex128)

        for site in range(N):
            ops = []
            for pos in range(N):
                if pos == site:
                    ops.append((sx, sy, sz))
                else:
                    ops.append((id2, id2, id2))

            sx_i = ops[0][0]
            sy_i = ops[0][1]
            sz_i = ops[0][2]
            for pos in range(1, N):
                sx_i = np.kron(sx_i, ops[pos][0])
                sy_i = np.kron(sy_i, ops[pos][1])
                sz_i = np.kron(sz_i, ops[pos][2])

            Jx += 0.5 * sx_i
            Jy += 0.5 * sy_i
            Jz += 0.5 * sz_i

        return Jx, Jy, Jz

    @staticmethod
    def _build_collective_operators_symmetric(N):
        J = N / 2.0
        dim = int(2 * J + 1)

        M_vals = np.arange(J, -J - 1, -1, dtype=float)

        Jp = np.zeros((dim, dim), dtype=np.complex128)
        Jm = np.zeros((dim, dim), dtype=np.complex128)
        Jz = np.zeros((dim, dim), dtype=np.complex128)

        for i, M in enumerate(M_vals):
            Jz[i, i] = M

            if i > 0:
                coef = math.sqrt(J * (J + 1.0) - M * (M + 1.0))
                Jp[i - 1, i] = coef

            if i < dim - 1:
                coef = math.sqrt(J * (J + 1.0) - M * (M - 1.0))
                Jm[i + 1, i] = coef

        Jx = 0.5 * (Jp + Jm)
        Jy = -0.5j * (Jp - Jm)

        return Jx, Jy, Jz

N = 4
epsilon_lip = 1.0
k_levels = 3

lipkin = LipkinModel(N=N, epsilon=epsilon_lip,
                    use_symmetric_sector=True)

V_dyn = 1.0
H_dyn = lipkin.hamiltonian(V_dyn)

from numpy.linalg import eigh
evals_dyn, evecs_dyn = eigh(H_dyn)
psi0_true = evecs_dyn[:, 0]  # ground state

Jz = lipkin.Jz

T_max = 10.0
Nt = 200
times = np.linspace(0.0, T_max, Nt)

Jz_t_true = lipkin.observable_time_series(V_dyn, psi0_true, Jz, times).real

plt.figure(figsize=(8, 5))
plt.plot(times, Jz_t_true, lw=2)
plt.xlabel("t")
plt.ylabel("<Jz(t)>")
plt.title(f"Exact <Jz(t)> (Lipkin, V={V_dyn})")
plt.tight_layout()
plt.show()

"""##Train PMM on $<Jz>_t$"""

Jz_t = times.reshape(-1,1)
V = np.full((times.shape[0],1),V_dyn)

X = np.hstack((V,Jz_t))
y = Jz_t_true.reshape(-1,1)

X_train = X[:20]
y_train = y[:20]

x_val = None
y_val = None
n = 5
k = 1 #here we are setting Psi_0 to be the ground state of the PMM V[:,0]
l = 1
type = True #True: Complex

Model2 = model2(n,
              X_train,
              y_train,
              k,
              l,
              type,
              x_val,
              y_val,
              param=None,
              epochs = 1000,
              learning_rate =0.01,
              print_every = 100,
              verbose=True)

New_theta = Model2.train()

y_pred = Model2.predict(New_theta,X)
plt.figure(figsize=(8, 5))
plt.plot(times, Jz_t_true, lw=2, label="Exact <Jz(t)>")
plt.plot(times, y_pred, "--", lw=2, label="PMM <Jz(t)>")
#vertical line
plt.axvline(x=times[20], color='r', linestyle='--',label='training data')
plt.xlabel("t")
plt.ylabel("<Jz(t)>")
plt.title("Time evolution: exact vs PMM (AD)")
plt.legend()
plt.tight_layout()
plt.show()

#plot abs error
plt.figure(figsize=(8, 5))
plt.plot(times, np.abs(Jz_t_true.reshape(-1,) - y_pred.reshape(-1,)), lw=2)
plt.xlabel("t")
plt.ylabel("abs error")
plt.title("abs error: exact vs PMM (AD)")
plt.yscale('log')
plt.tight_layout()
plt.show()